<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Chapter 2: Working with Text Data

Packages that are being used in this notebook:

In [2]:
%pip install torch tiktoken

Note: you may need to restart the kernel to use updated packages.


In [3]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.10.0
tiktoken version: 0.12.0


- This chapter covers data preparation and sampling to get input data "ready" for the LLM

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" width="500px">

## 2.1 Understanding word embeddings

- No code in this section

- There are many forms of embeddings; we focus on text embeddings in this book

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/02.webp" width="500px">

- LLMs work with embeddings in high-dimensional spaces (i.e., thousands of dimensions)
- Since we can't visualize such high-dimensional spaces (we humans think in 1, 2, or 3 dimensions), the figure below illustrates a 2-dimensional embedding space

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/03.webp" width="300px">

## 2.2 Tokenizing text

- In this section, we tokenize text, which means breaking text into smaller units, such as individual words and punctuation characters

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="300px">

- Load raw text we want to work with
- [The Verdict by Edith Wharton](https://en.wikisource.org/wiki/The_Verdict) is a public domain short story

In [4]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)


# The book originally used the following code below
# However, urllib uses older protocol settings that
# can cause problems for some readers using a VPN.
# The `requests` version above is more robust
# in that regard.

"""
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)
"""

'\nimport os\nimport urllib.request\n\nif not os.path.exists("the-verdict.txt"):\n    url = ("https://raw.githubusercontent.com/rasbt/"\n           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"\n           "the-verdict.txt")\n    file_path = "the-verdict.txt"\n    urllib.request.urlretrieve(url, file_path)\n'

<br>

---

<br>

#### Troubleshooting SSL certificate errors

- Some readers reported seeing ssl.SSLCertVerificationError: `SSL: CERTIFICATE_VERIFY_FAILED` when running `urllib.request.urlretrieve` in VSCode or Jupyter. 
- This usually means Python's certificate bundle is outdated.


**Fixes**

- Use Python ≥ 3.9; you can check your Python version by executing the following code:
```python
import sys
print(sys.__version__)
```
- Upgrade the cert bundle:
  - pip: `pip install --upgrade certifi`
  - uv: `uv pip install --upgrade certifi`
- Restart the Jupyter kernel after upgrading.
- If you still encounter an `ssl.SSLCertVerificationError` when executing the previous code cell, please see the discussion at [more information here on GitHub](https://github.com/rasbt/LLMs-from-scratch/pull/403)

<br>

---

<br>

In [5]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- The goal is to tokenize and embed this text for an LLM
- Let's develop a simple tokenizer based on some simple sample text that we can then later apply to the text above
- The following regular expression will split on whitespaces

In [6]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


- We don't only want to split on whitespaces but also commas and periods, so let's modify the regular expression to do that as well

In [7]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


- As we can see, this creates empty strings, let's remove them

In [8]:
# Strip whitespace from each item and then filter out any empty strings.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


- This looks pretty good, but let's also handle other types of punctuation, such as periods, question marks, and so on

In [9]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


- This is pretty good, and we are now ready to apply this tokenization to the raw text

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/05.webp" width="350px">

In [10]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


- Let's calculate the total number of tokens

In [11]:
print(len(preprocessed))

4690


> ## ⬤ Explicación - Tokenización



La tokenización convierte texto en unidades discretas (tokens) usando expresiones regulares que separan tanto espacios como signos de puntuación. El regex `r'([,.:;?_!"()\']|--|\s)'` divide el texto en palabras individuales Y símbolos únicos.


Este paso es muy importante porque las redes neuronales solo pueden procesar números, no texto. La tokenización es el primer paso obligatorio para transformar lenguaje humano en datos procesables. Sin no se tiene una tokenizacion correcta, todo el pipeline fallaria.

En este paso de la tokenización se valida:

- Que cada signo de puntuación se trate como token independiente ("test." → ["test", "."])
- Que los espacios se preserven inicialmente para luego ser eliminados
- Que el proceso sea determinístico: la misma frase siempre produce los mismos tokens



Si no se separara la puntuación correctamente, el vocabulario se expandiría mucho : "test", "test.", "test,", "test!", etc. serían tokens diferentes, multiplicando el tamaño del vocabulario innecesariamente. Con un vocabulario de 50K palabras, esto podría crecer a 200K+ combinaciones.

Si la tokenización no fuera determinística, el modelo nunca aprendería patrones consistentes. Durante el entrenamiento, la misma oración se tokenizaría diferente en cada epoch, haciendo imposible la convergencia.

## 2.3 Converting tokens into token IDs

- Next, we convert the text tokens into token IDs that we can process via embedding layers later

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="500px">

- From these tokens, we can now build a vocabulary that consists of all the unique tokens

In [12]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [13]:
vocab = {token:integer for integer,token in enumerate(all_words)}

- Below are the first 50 entries in this vocabulary:

In [14]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


> ## ⬤ Explicación - Conversión a IDs



En este paso se  crea un diccionario que mapea cada token único a un ID numérico entero. El vocabulario se construye ordenando alfabéticamente todos los tokens únicos y asignándoles índices secuenciales: `vocab = {token:integer for integer,token in enumerate(all_words)}`.


Es necesario este mapeo porque las operaciones de redes neuronales requieren números, no strings. Los IDs permiten indexar en matrices de embeddings y calcular gradientes durante backpropagation. No se puede calcular `∂Loss/∂"hello"`, pero sí `∂Loss/∂embedding_matrix[ID]`.

En esta paso se valida lo siguiente:

- Que cada token único tenga exactamente un ID (mapeo biyectivo)
- Que el ID sea un entero válido para indexación en arrays
- Que el vocabulario cubra todos los tokens del dataset
- Que el mapeo inverso (ID → token) sea posible para decodificación

**¿Qué pasaría si no se hiciera correctamente este paso?**

Si dos tokens diferentes tuvieran el mismo ID, el modelo los trataría como idénticos semánticamente. "banco" (institución financiera) y "banco" (asiento) tendrían el mismo embedding, perdiendo diferenciación contextual.

Si los IDs no fueran consistentes entre ejecuciones, el modelo preentrenado sería incompatible con nuevos datos. Un modelo entrenado donde "hello" = 1159 fallaría si en producción "hello" = 2500.

Si faltaran tokens en el vocabulario (<|unk|>), el modelo no podría procesar palabras nuevas, limitando severamente su capacidad de generalización. Por eso BPE (Byte Pair Encoding) es superior: descompone palabras desconocidas en subpalabras conocidas.

Este paso se relaciona con **embedding lookup**: el ID es el índice que se usa para extraer el vector correspondiente de la matriz de embeddings. Si los IDs no estan bien definidos, no se puede acceder a las representaciones vectoriales.

---

- Below, we illustrate the tokenization of a short sample text using a small vocabulary:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="500px">

- Putting it now all together into a tokenizer class

In [15]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

- The `encode` function turns text into token IDs
- The `decode` function turns token IDs back into text

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="500px">

- We can use the tokenizer to encode (that is, tokenize) texts into integers
- These integers can then be embedded (later) as input of/for the LLM

In [16]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


- We can decode the integers back into text

In [17]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [18]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## 2.4 Adding special context tokens

- It's useful to add some "special" tokens for unknown words and to denote the end of a text

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="500px">

- Some tokenizers use special tokens to help the LLM with additional context
- Some of these special tokens are
  - `[BOS]` (beginning of sequence) marks the beginning of text
  - `[EOS]` (end of sequence) marks where the text ends (this is usually used to concatenate multiple unrelated texts, e.g., two different Wikipedia articles or two different books, and so on)
  - `[PAD]` (padding) if we train LLMs with a batch size greater than 1 (we may include multiple texts with different lengths; with the padding token we pad the shorter texts to the longest length so that all texts have an equal length)
- `[UNK]` to represent words that are not included in the vocabulary

- Note that GPT-2 does not need any of these tokens mentioned above but only uses an `<|endoftext|>` token to reduce complexity
- The `<|endoftext|>` is analogous to the `[EOS]` token mentioned above
- GPT also uses the `<|endoftext|>` for padding (since we typically use a mask when training on batched inputs, we would not attend padded tokens anyways, so it does not matter what these tokens are)
- GPT-2 does not use an `<UNK>` token for out-of-vocabulary words; instead, GPT-2 uses a byte-pair encoding (BPE) tokenizer, which breaks down words into subword units which we will discuss in a later section



- We use the `<|endoftext|>` tokens between two independent sources of text:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="500px">

- Let's see what happens if we tokenize the following text:

In [19]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

- The above produces an error because the word "Hello" is not contained in the vocabulary
- To deal with such cases, we can add special tokens like `"<|unk|>"` to the vocabulary to represent unknown words
- Since we are already extending the vocabulary, let's add another token called `"<|endoftext|>"` which is used in GPT-2 training to denote the end of a text (and it's also used between concatenated text, like if our training datasets consists of multiple articles, books, etc.)

In [20]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [21]:
len(vocab.items())

1132

In [22]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


- We also need to adjust the tokenizer accordingly so that it knows when and how to use the new `<unk>` token

In [23]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Let's try to tokenize text with the modified tokenizer:

In [24]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [25]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [26]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

## 2.5 BytePair encoding

- GPT-2 used BytePair encoding (BPE) as its tokenizer
- it allows the model to break down words that aren't in its predefined vocabulary into smaller subword units or even individual characters, enabling it to handle out-of-vocabulary words
- For instance, if GPT-2's vocabulary doesn't have the word "unfamiliarword," it might tokenize it as ["unfam", "iliar", "word"] or some other subword breakdown, depending on its trained BPE merges
- The original BPE tokenizer can be found here: [https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- In this chapter, we are using the BPE tokenizer from OpenAI's open-source [tiktoken](https://github.com/openai/tiktoken) library, which implements its core algorithms in Rust to improve computational performance
- I created a notebook in the [./bytepair_encoder](../02_bonus_bytepair-encoder) that compares these two implementations side-by-side (tiktoken was about 5x faster on the sample text)

In [ ]:
# pip install tiktoken

In [27]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


In [28]:
tokenizer = tiktoken.get_encoding("gpt2")

In [29]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [30]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- BPE tokenizers break down unknown words into subwords and individual characters:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="300px">

## 2.6 Data sampling with a sliding window

- We train LLMs to generate one word at a time, so we want to prepare the training data accordingly where the next word in a sequence represents the target to predict:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

In [31]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


- For each text chunk, we want the inputs and targets
- Since we want the model to predict the next word, the targets are the inputs shifted by one position to the right

In [32]:
enc_sample = enc_text[50:]

In [33]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


- One by one, the prediction would look like as follows:

In [34]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [35]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


- We will take care of the next-word prediction in a later chapter after we covered the attention mechanism
- For now, we implement a simple data loader that iterates over the input dataset and returns the inputs and targets shifted by one

- Install and import PyTorch (see Appendix A for installation tips)

In [36]:
import torch
print("PyTorch version:", torch.__version__)

c:\Users\eliza\OneDrive\Documentos\ECI\8_SEMESTRE\TDSE\PrimerCorte\LLM-text-preprocessing-embeddings\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


PyTorch version: 2.10.0+cpu


- We use a sliding window approach, changing the position by +1:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="500px">

- Create dataset and dataloader that extract chunks from the input text dataset

In [37]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [38]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

> ## ⬤ Explicación - Sliding Window


En este paso el sliding window crea pares de secuencias (input, target) donde el target está desplazado una posición adelante. Con `stride`, se controla cuánto se mueve la ventana. Por ejemplo, para `max_length=4` y `stride=2`:
- Ventana 1: input=[0,1,2,3] → target=[1,2,3,4]  
- Ventana 2: input=[2,3,4,5] → target=[3,4,5,6]  
- Ventana 3: input=[4,5,6,7] → target=[5,6,7,8]



Este paso es importante para el entrenamiento porque los LLMs aprenden mediante next-token prediction: ver los primeros N tokens y predecir el token N+1. El sliding window maximiza los ejemplos de entrenamiento extraídos del texto. Con stride=1 en un texto de 5000 tokens, se obtienen ~5000 ejemplos; con stride=max_length (sin overlap), solo ~20 ejemplos.

En este paso se valida lo siguiente:

- Que `target_chunk` empiece exactamente en `i+1` (shifted by one)
- Que la longitud de input y target sean iguales (`max_length`)
- Que `stride` no sea cero (evitar bucle infinito)
- Que el texto tenga suficientes tokens (`len(token_ids) > max_length`)

Si no se implementa correctamente podrian pasar las siguientes situaciones:

Si el target no estuviera shifted (+1), el modelo aprendería a copiar el input, no a predecir el siguiente token. Entraría en un modo de memorización trivial.

Si stride = max_length (sin overlap), se desperdiciarían datos. De "The cat sat on the mat" solo se extraería un ejemplo, cuando podrían ser 5-6 ventanas sobrelapadas con diferentes contextos.

Si stride < 1 o stride > max_length, o habría bucles infinitos o se saltarían secciones del texto, causando que el modelo nunca vea ciertas secuencias del dataset.


Este mecanismo replica exactamente cómo el modelo funcionará en inferencia. Al generar texto, el modelo predice token por token, usando todos los tokens anteriores como contexto. Si durante el entrenamiento no practicara esto con sliding windows de diferentes longitudes, no aprendería las dependencias secuenciales del lenguaje.



---


- Let's test the dataloader with a batch size of 1 for an LLM with a context size of 4:

In [39]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [40]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [41]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


- An example using stride equal to the context length (here: 4) as shown below:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="500px">

- We can also create batched outputs
- Note that we increase the stride here so that we don't have overlaps between the batches, since more overlap could lead to increased overfitting

In [42]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## 2.7 Creating token embeddings

- The data is already almost ready for an LLM
- But lastly let us embed the tokens in a continuous vector representation using an embedding layer
- Usually, these embedding layers are part of the LLM itself and are updated (trained) during model training

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="400px">

- Suppose we have the following four input examples with input ids 2, 3, 5, and 1 (after tokenization):

In [43]:
input_ids = torch.tensor([2, 3, 5, 1])

- For the sake of simplicity, suppose we have a small vocabulary of only 6 words and we want to create embeddings of size 3:

In [44]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- This would result in a 6x3 weight matrix:

In [45]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


- For those who are familiar with one-hot encoding, the embedding layer approach above is essentially just a more efficient way of implementing one-hot encoding followed by matrix multiplication in a fully-connected layer, which is described in the supplementary code in [./embedding_vs_matmul](../03_bonus_embedding-vs-matmul)
- Because the embedding layer is just a more efficient implementation that is equivalent to the one-hot encoding and matrix-multiplication approach it can be seen as a neural network layer that can be optimized via backpropagation

- To convert a token with id 3 into a 3-dimensional vector, we do the following:

In [46]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


- Note that the above is the 4th row in the `embedding_layer` weight matrix
- To embed all four `input_ids` values above, we do

In [47]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


- An embedding layer is essentially a look-up operation:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="500px">

- **You may be interested in the bonus content comparing embedding layers with regular linear layers: [../03_bonus_embedding-vs-matmul](../03_bonus_embedding-vs-matmul)**

## 2.8 Encoding word positions

- Embedding layer convert IDs into identical vector representations regardless of where they are located in the input sequence:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="400px">

- Positional embeddings are combined with the token embedding vector to form the input embeddings for a large language model:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="500px">

- The BytePair encoder has a vocabulary size of 50,257:
- Suppose we want to encode the input tokens into a 256-dimensional vector representation:

In [48]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- If we sample data from the dataloader, we embed the tokens in each batch into a 256-dimensional vector
- If we have a batch size of 8 with 4 tokens each, this results in a 8 x 4 x 256 tensor:

In [49]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [50]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [53]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(token_embeddings)

torch.Size([8, 4, 256])
tensor([[[ 0.4913,  1.1239,  1.4588,  ..., -0.3995, -1.8735, -0.1445],
         [ 0.4481,  0.2536, -0.2655,  ...,  0.4997, -1.1991, -1.1844],
         [-0.2507, -0.0546,  0.6687,  ...,  0.9618,  2.3737, -0.0528],
         [ 0.9457,  0.8657,  1.6191,  ..., -0.4544, -0.7460,  0.3483]],

        [[ 1.5460,  1.7368, -0.7848,  ..., -0.1004,  0.8584, -0.3421],
         [-1.8622, -0.1914, -0.3812,  ...,  1.1220, -0.3496,  0.6091],
         [ 1.9847, -0.6483, -0.1415,  ..., -0.3841, -0.9355,  1.4478],
         [ 0.9647,  1.2974, -1.6207,  ...,  1.1463,  1.5797,  0.3969]],

        [[-0.7713,  0.6572,  0.1663,  ..., -0.8044,  0.0542,  0.7426],
         [ 0.8046,  0.5047,  1.2922,  ...,  1.4648,  0.4097,  0.3205],
         [ 0.0795, -1.7636,  0.5750,  ...,  2.1823,  1.8231, -0.3635],
         [ 0.4267, -0.0647,  0.5686,  ..., -0.5209,  1.3065,  0.8473]],

        ...,

        [[-1.6156,  0.9610, -2.6437,  ..., -0.9645,  1.0888,  1.6383],
         [-0.3985, -0.9235, -1.31

- GPT-2 uses absolute position embeddings, so we just create another embedding layer:

In [54]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
print(pos_embedding_layer.weight)

Parameter containing:
tensor([[-0.8194,  0.5543, -0.8290,  ...,  0.1325,  0.2115,  0.3610],
        [ 0.4193, -0.9461, -0.3407,  ...,  0.7930,  1.7009,  0.5663],
        [-0.2362, -1.7187, -1.0489,  ...,  1.1218,  0.2796,  0.9912],
        [-0.9549,  0.4699,  0.2580,  ..., -1.3689,  1.6505,  1.3488]],
       requires_grad=True)


In [55]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(pos_embeddings)

torch.Size([4, 256])
tensor([[-0.8194,  0.5543, -0.8290,  ...,  0.1325,  0.2115,  0.3610],
        [ 0.4193, -0.9461, -0.3407,  ...,  0.7930,  1.7009,  0.5663],
        [-0.2362, -1.7187, -1.0489,  ...,  1.1218,  0.2796,  0.9912],
        [-0.9549,  0.4699,  0.2580,  ..., -1.3689,  1.6505,  1.3488]],
       grad_fn=<EmbeddingBackward0>)


- To create the input embeddings used in an LLM, we simply add the token and the positional embeddings:

In [56]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(input_embeddings)

torch.Size([8, 4, 256])
tensor([[[-0.3281,  1.6782,  0.6298,  ..., -0.2670, -1.6620,  0.2165],
         [ 0.8674, -0.6925, -0.6063,  ...,  1.2927,  0.5018, -0.6181],
         [-0.4869, -1.7733, -0.3802,  ...,  2.0836,  2.6533,  0.9384],
         [-0.0091,  1.3356,  1.8771,  ..., -1.8233,  0.9045,  1.6972]],

        [[ 0.7267,  2.2912, -1.6138,  ...,  0.0321,  1.0699,  0.0189],
         [-1.4429, -1.1375, -0.7219,  ...,  1.9150,  1.3513,  1.1754],
         [ 1.7486, -2.3669, -1.1904,  ...,  0.7377, -0.6559,  2.4390],
         [ 0.0099,  1.7672, -1.3627,  ..., -0.2226,  3.2302,  1.7457]],

        [[-1.5907,  1.2115, -0.6627,  ..., -0.6719,  0.2657,  1.1036],
         [ 1.2239, -0.4414,  0.9515,  ...,  2.2578,  2.1106,  0.8868],
         [-0.1567, -3.4823, -0.4740,  ...,  3.3041,  2.1027,  0.6277],
         [-0.5282,  0.4051,  0.8265,  ..., -1.8898,  2.9570,  2.1961]],

        ...,

        [[-2.4349,  1.5153, -3.4727,  ..., -0.8320,  1.3004,  1.9994],
         [ 0.0208, -1.8696, -1.65

- In the initial phase of the input processing workflow, the input text is segmented into separate tokens
- Following this segmentation, these tokens are transformed into token IDs based on a predefined vocabulary:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="400px">

---

> ## ⬤ Explicación -  ¿Por qué los embeddings codifican significado y cómo se relacionan con redes neuronales?

### El problema que resuelven los embeddings

Los token IDs son números discretos arbitrarios. Que "cat"=2936 y "dog"=3290 no implica ninguna relación semántica. La distancia numérica |2936-3290|=354 no significa nada. Estos IDs solo sirven como índices.

### ¿Qué son los embeddings?

Un embedding layer (`nn.Embedding(vocab_size, embedding_dim)`) es una **matriz de pesos entrenable** de tamaño [50257 x 256]. Cada fila es un vector que representa un token. Esta matriz es un parámetro del modelo que se optimiza mediante backpropagation.

Cuando se indexa `embedding_matrix[token_id]`, se extrae un vector denso de 256 dimensiones. Este vector es la representación del token en un espacio continuo.

### ¿Cómo empieza el modelo a entender el significado cuando se entrena?

**Inicio:** Los embeddings se inicializan aleatoriamente y no tienen significado.

**Durante entrenamiento:** El modelo debe predecir la siguiente palabra basándose en el contexto. Si las frases son:
- "The **cat** meows"  
- "The **dog** barks"  
- "The **cat** sleeps"

El modelo ajusta los embeddings para minimizar el error de predicción. Palabras que aparecen en contextos similares (cat y dog ambos aparecen después de "The" y antes de verbos de acción) necesitan tener embeddings similares para que el modelo haga predicciones coherentes.

**Resultado:** Después de billones de actualizaciones, embeddings de palabras semánticamente relacionadas convergen a vectores cercanos en el espacio latente. Esto es la Distributional Hypothesis que son palabras con significados similares que aparecen en contextos similares.

### Validación matemática de similitud semántica

Con embeddings entrenados, se puede medir similitud con cosine similarity:

```
cosine_similarity(cat, dog) ≈ 0.85  (muy similar)
cosine_similarity(cat, algorithm) ≈ 0.12  (poco similar)
```

Incluso relaciones analógicas pueden surgir:
```
king - man + woman ≈ queen
```

Esto NO fue programado explícitamente. Emerge del proceso de optimización.

### Relación con conceptos de redes neuronales

**1. Los embeddings son pesos entrenables como con cualquier otra capa**

Durante backpropagation:
```
Forward: x = embedding_matrix[token_id]
Loss: L = loss_function(prediction, truth)
Backward: ∂L/∂embedding_matrix[token_id] 
Update: embedding_matrix[token_id] -= lr * gradient
```

La diferencia con una capa linear es que el embedding layer hace indexing en lugar de multiplicación matricial completa, pero es matemáticamente equivalente a una linear layer con input one-hot.

**2. Espacio latente de menor dimensión**

One-hot encoding: vectores de [50257] dimensiones, 99.998% ceros.  
Embeddings: vectores de [256] dimensiones, todo valores no-cero.

Esto es compresión dimensional similar a autoencoders. Se proyecta información de un espacio enorme y disperso a un espacio denso donde la distancia euclidiana tiene significado semántico.

**3. Continuidad permite interpolación**

A diferencia de IDs discretos, el espacio de embeddings es continuo. Vectores intermedios entre dos palabras tienen significado. Esto permite al modelo extrapolar a conceptos que nunca vio explícitamente.

**4. Composicionalidad con otros embeddings**

```python
input_embeddings = token_embeddings + positional_embeddings
```

Se combinan múltiples fuentes de información (qué palabra + dónde está) en una sola representación. Las capas superiores (attention, feedforward) procesan estas representaciones compuestas.

**5. Gradientes fluyen a través de toda la arquitectura**

```
Loss → Output layer → Attention → Embeddings
```

Cuando el modelo se equivoca, el error backpropaga hasta los embeddings. Palabras que causan errores sistemáticos se ajustan para reducir confusión.

### ¿Qué pasaría si no se usaran embeddings?

**Si se usara one-hot:**
- Vectores de 50K dimensiones → ineficiente en memoria y cómputo
- Sin noción de similitud semántica → el modelo no generalizaría
- Gradientes dispersos (99.998% ceros) → aprendizaje lento

**Si los embeddings no fueran entrenables:**
- Representaciones fijas no capturarían matices del dataset
- No habría adaptación durante fine-tuning
- El modelo sería incapaz de mejorar su comprensión




Cuando un agente LLM procesa una instrucción como "deploy the application", los embeddings de "deploy" estarán cerca de "launch", "start", "activate". El modelo puede inferir intención correcta incluso con variaciones en el fraseo.

Para razonamiento multi-paso:
```
Instrucción: "First install dependencies, then compile the code"
```

Los embeddings de "first...then" codifican orden temporal. El modelo aprendió que tokens como "before", "after", "next" se agrupan en regiones del espacio vectorial asociadas con secuencialidad.

### Conclusión

Los embeddings son la capa que convierte palabras o tokens (símbolos discretos) en vectores numéricos que la red neuronal puede procesar. Son el puente entre el lenguaje y el espacio matemático donde opera el modelo.

Estas representaciones no se definen a mano. El modelo las aprende automáticamente durante el entrenamiento. Ajusta los vectores para reducir el error de predicción. Por eso, palabras que aparecen en contextos parecidos (como “cat” y “dog”) terminan con vectores cercanos, sin que nadie lo programe explícitamente.

Si los embeddings no captaran relaciones de significado, el modelo solo manipularía símbolos sin estructura. Los embeddings permiten que el LLM generalice, relacione conceptos y vaya más allá de simples coincidencias de texto.

---

> ##  Experiment: Impact of `max_length` and `stride` on Dataset Size

En esta sección experimental se explora cómo los parámetros `max_length` y `stride` afectan la cantidad de muestras de entrenamiento generadas y por qué el overlap (solapamiento) es beneficioso para el aprendizaje del modelo.

### Experiment Setup

Vamos a probar diferentes combinaciones de `max_length` y `stride` y observar cuántas muestras se generan del mismo texto.

In [62]:
def count_samples(max_length, stride):
    """
    Count how many samples are generated with the given parameters
    """
    dataloader = create_dataloader_v1(
        raw_text, 
        batch_size=1,  # batch_size doesn't affect total count
        max_length=max_length,
        stride=stride,
        shuffle=False,
        drop_last=False  # Don't discard last batch
    )
    return len(dataloader.dataset)

# Experiments with different configurations
experiments = [
    # (max_length, stride, description)
    (4, 4, "No overlap - stride = max_length"),
    (4, 2, "50% overlap - stride = max_length/2"),
    (4, 1, "Maximum overlap - stride = 1"),
    (8, 8, "No overlap - larger context"),
    (8, 4, "50% overlap - larger context"),
    (16, 16, "No overlap - much larger context"),
    (16, 4, "75% overlap - much larger context"),
]

print(f"Total text has {len(enc_text)} tokens\n")
print("="*80)
print(f"{'max_length':<12} | {'stride':<8} | {'samples':<10} | {'description':<30}")
print("="*80)

for max_len, stride_val, desc in experiments:
    num_samples = count_samples(max_len, stride_val)
    print(f"{max_len:<12} | {stride_val:<8} | {num_samples:<10} | {desc:<30}")

print("="*80)

Total text has 5145 tokens

max_length   | stride   | samples    | description                   
4            | 4        | 1286       | No overlap - stride = max_length
4            | 2        | 2571       | 50% overlap - stride = max_length/2
4            | 1        | 5141       | Maximum overlap - stride = 1  
8            | 8        | 643        | No overlap - larger context   
8            | 4        | 1285       | 50% overlap - larger context  
16           | 16       | 321        | No overlap - much larger context
16           | 4        | 1283       | 75% overlap - much larger context


### Visual Overlap Analysis

Visualización de cómo se solapan las ventanas de entrenamiento con diferentes configuraciones.

In [64]:
def visualize_overlap(max_length, stride, num_windows=5):
    """
    Visualize how training windows overlap
    """
    print(f"\n Visualization: max_length={max_length}, stride={stride}")
    print("-" * 60)
    
    sample_tokens = list(range(20))  # Example tokens [0,1,2,...,19]
    
    for i in range(num_windows):
        start = i * stride
        end = start + max_length
        
        if end > len(sample_tokens):
            break
            
        window = sample_tokens[start:end]
        
        # Create visual representation
        visual = ['.'] * len(sample_tokens)
        for idx in range(start, end):
            visual[idx] = '█'
        
        print(f"Window {i+1}: {''.join(visual)} → tokens {window}")

# Compare different strategies
visualize_overlap(max_length=4, stride=4)  # No overlap
visualize_overlap(max_length=4, stride=2)  # 50% overlap
visualize_overlap(max_length=4, stride=1)  # Maximum overlap


 Visualization: max_length=4, stride=4
------------------------------------------------------------
Window 1: ████................ → tokens [0, 1, 2, 3]
Window 2: ....████............ → tokens [4, 5, 6, 7]
Window 3: ........████........ → tokens [8, 9, 10, 11]
Window 4: ............████.... → tokens [12, 13, 14, 15]
Window 5: ................████ → tokens [16, 17, 18, 19]

 Visualization: max_length=4, stride=2
------------------------------------------------------------
Window 1: ████................ → tokens [0, 1, 2, 3]
Window 2: ..████.............. → tokens [2, 3, 4, 5]
Window 3: ....████............ → tokens [4, 5, 6, 7]
Window 4: ......████.......... → tokens [6, 7, 8, 9]
Window 5: ........████........ → tokens [8, 9, 10, 11]

 Visualization: max_length=4, stride=1
------------------------------------------------------------
Window 1: ████................ → tokens [0, 1, 2, 3]
Window 2: .████............... → tokens [1, 2, 3, 4]
Window 3: ..████.............. → tokens [2, 3, 4,

### Data Efficiency Calculation

Cálculo de qué porcentaje del texto se ve en cada epoch y cuánto overlap existe.

In [66]:
def calculate_data_efficiency(max_length, stride):
    """
    Calculate what percentage of text is seen in each epoch
    """
    total_tokens = len(enc_text)
    num_samples = count_samples(max_length, stride)
    
    # Unique tokens seen (approximation)
    tokens_covered = min(stride * num_samples + max_length, total_tokens)
    
    # Total tokens processed (counting repetitions)
    tokens_processed = num_samples * max_length
    
    # Overlap factor
    overlap_factor = tokens_processed / tokens_covered if tokens_covered > 0 else 0
    
    return {
        'num_samples': num_samples,
        'tokens_covered': tokens_covered,
        'tokens_processed': tokens_processed,
        'coverage_percent': (tokens_covered / total_tokens) * 100,
        'overlap_factor': overlap_factor,
        'efficiency': num_samples / (total_tokens / max_length)
    }

# Analyze efficiency
configs = [
    (4, 4),   # No overlap
    (4, 2),   # 50% overlap
    (4, 1),   # Maximum overlap
    (8, 4),   # 50% overlap, larger context
]

print("\n Data Efficiency Analysis")
print("="*100)
print(f"{'max_len':<8} | {'stride':<8} | {'samples':<10} | {'coverage%':<12} | {'overlap_x':<10} | {'efficiency':<10}")
print("="*100)

for max_len, stride_val in configs:
    stats = calculate_data_efficiency(max_len, stride_val)
    print(f"{max_len:<8} | {stride_val:<8} | {stats['num_samples']:<10} | "
          f"{stats['coverage_percent']:<12.1f} | {stats['overlap_factor']:<10.2f} | "
          f"{stats['efficiency']:<10.2f}")

print("="*100)


 Data Efficiency Analysis
max_len  | stride   | samples    | coverage%    | overlap_x  | efficiency
4        | 4        | 1286       | 100.0        | 1.00       | 1.00      
4        | 2        | 2571       | 100.0        | 2.00       | 2.00      
4        | 1        | 5141       | 100.0        | 4.00       | 4.00      
8        | 4        | 1285       | 100.0        | 2.00       | 2.00      


### Practical Demonstration: Context Learning

Ejemplos concretos de qué aprende el modelo con diferentes overlaps.

In [69]:
def demonstrate_context_learning(max_length, stride):
    """
    Show concrete examples of what the model learns with different overlaps
    """
    dataloader = create_dataloader_v1(
        raw_text, 
        batch_size=1,
        max_length=max_length,
        stride=stride,
        shuffle=False,
        drop_last=False
    )
    
    print(f"\n Training Examples (max_length={max_length}, stride={stride}):")
    print("-" * 80)
    
    # Show first 5 samples
    for i, (inputs, targets) in enumerate(dataloader):
        if i >= 5:
            break
        
        input_text = tokenizer.decode(inputs[0].tolist())
        target_text = tokenizer.decode(targets[0].tolist())
        
        print(f"\nSample {i+1}:")
        print(f"  Input:  {input_text[:60]}...")
        print(f"  Target: {target_text[:60]}...")

# Compare what it learns with and without overlap
print("\n" + ""*80)
print("WITHOUT OVERLAP (stride = max_length)")
print(""*80)
demonstrate_context_learning(max_length=8, stride=8)

print("\n" + ""*80)
print("WITH 50% OVERLAP (stride = max_length/2)")
print(""*80)
demonstrate_context_learning(max_length=8, stride=4)



WITHOUT OVERLAP (stride = max_length)


 Training Examples (max_length=8, stride=8):
--------------------------------------------------------------------------------

Sample 1:
  Input:  I HAD always thought Jack Gis...
  Target:  HAD always thought Jack Gisburn...

Sample 2:
  Input:  burn rather a cheap genius--though a...
  Target:  rather a cheap genius--though a good...

Sample 3:
  Input:   good fellow enough--so it was no...
  Target:  fellow enough--so it was no great...

Sample 4:
  Input:   great surprise to me to hear that,...
  Target:  surprise to me to hear that, in...

Sample 5:
  Input:   in the height of his glory, he...
  Target:  the height of his glory, he had...


WITH 50% OVERLAP (stride = max_length/2)


 Training Examples (max_length=8, stride=4):
--------------------------------------------------------------------------------

Sample 1:
  Input:  I HAD always thought Jack Gis...
  Target:  HAD always thought Jack Gisburn...

Sample 2:
  Input:   thought Jack 

### Impact on Prediction Examples

Demostración de por qué el overlap ayuda en las predicciones.

In [72]:
def show_prediction_examples():
    """
    Show how overlap helps the model learn better predictions
    """
    print("\n How does overlap help predictions?")
    print("="*80)
    
    # Simulation of what the model sees
    test_sentence = "The quick brown fox jumps over the lazy dog"
    tokens = tokenizer.encode(test_sentence)
    
    print(f"\nSentence: '{test_sentence}'")
    print(f"Tokens: {tokens}\n")
    
    # Without overlap (stride = max_length)
    print(" WITHOUT OVERLAP (stride=4, max_length=4):")
    for i in range(0, len(tokens)-4, 4):
        context = tokens[i:i+4]
        target = tokens[i+4] if i+4 < len(tokens) else "N/A"
        print(f"  {tokenizer.decode(context)} → predicts → {tokenizer.decode([target]) if target != 'N/A' else 'N/A'}")
    
    # With overlap (stride = 2)
    print("\n WITH OVERLAP (stride=2, max_length=4):")
    for i in range(0, len(tokens)-4, 2):
        context = tokens[i:i+4]
        target = tokens[i+4] if i+4 < len(tokens) else "N/A"
        print(f"  {tokenizer.decode(context)} → predicts → {tokenizer.decode([target]) if target != 'N/A' else 'N/A'}")
    
    print("\n Observation:")
    print("With overlap, the model sees MORE contexts to predict the same word.")
    print("This improves its ability to generalize linguistic patterns.")

show_prediction_examples()


 How does overlap help predictions?

Sentence: 'The quick brown fox jumps over the lazy dog'
Tokens: [464, 2068, 7586, 21831, 18045, 625, 262, 16931, 3290]

 WITHOUT OVERLAP (stride=4, max_length=4):
  The quick brown fox → predicts →  jumps
   jumps over the lazy → predicts →  dog

 WITH OVERLAP (stride=2, max_length=4):
  The quick brown fox → predicts →  jumps
   brown fox jumps over → predicts →  the
   jumps over the lazy → predicts →  dog

 Observation:
With overlap, the model sees MORE contexts to predict the same word.
This improves its ability to generalize linguistic patterns.


---

##  Conclusiones del Experimento

### Hallazgos Clave

**1. Relación entre stride y número de muestras:**

La fórmula aproximada es:
```
num_samples ≈ (total_tokens - max_length) / stride
```

- **stride = max_length** (sin overlap): Mínimo número de muestras, máxima eficiencia computacional
- **stride = 1** (máximo overlap): Máximo número de muestras, máximo costo computacional
- **stride = max_length/2**: Balance común entre datos y eficiencia

**2. Por qué el overlap es útil:**

**a) Contextos múltiples:**
- Sin overlap: Cada token aparece en exactamente UN contexto
- Con overlap: Cada token aparece en MÚLTIPLES contextos diferentes
- Ejemplo: La palabra "the" aparece después de "cat" en una ventana y después de "dog" en otra

**b) Aprendizaje de dependencias:**
- Con stride=1, el modelo ve TODAS las transiciones token-a-token
- Con stride=max_length, el modelo se pierde transiciones entre ventanas
- Crítico para aprender patrones como "not only... but also"

**c) Generalización mejorada:**
- Más ejemplos = menos overfitting
- El modelo aprende que las mismas palabras pueden aparecer en múltiples contextos
- Ejemplo práctico:
  - Sin overlap: "cat" solo se ve en "The cat meows"
  - Con overlap: "cat" se ve en "The cat", "cat meows", "meows loudly"

**3. Trade-offs:**

| Aspecto | Sin Overlap | Con 50% Overlap | Máximo Overlap |
|---------|------------|-----------------|----------------|
| Muestras | Mínimo | Medio | Máximo |
| Tiempo entrenamiento | Rápido | Medio | Lento |
| Calidad modelo | Básica | Buena | Excelente |
| Uso memoria | Bajo | Medio | Alto |
| Overfitting risk | Alto | Medio | Bajo |

**4. Recomendación práctica:**

Para la mayoría de casos, **stride = max_length / 2** es óptimo:
- Duplica las muestras vs sin overlap
- Costo computacional razonable
- Cada token se ve en 2 contextos diferentes
- Balance entre velocidad y calidad

**5. Impacto en sistemas agénticos:**

Un agente que ejecuta comandos necesita entender contexto:
- "delete the file" vs "do not delete the file"
- Sin overlap suficiente, podría perder el "not" si está en otra ventana
- Con overlap, aprende la negación completa en una sola secuencia

### Observaciones del Experimento

Al ejecutar el código anterior con el texto de ejemplo (~5145 tokens):

- Con max_length=4, stride=4: ~1286 muestras (cada token visto 1 vez)
- Con max_length=4, stride=2: ~2570 muestras (cada token visto ~2 veces)  
- Con max_length=4, stride=1: ~5141 muestras (cada token visto ~4 veces)

Esto demuestra que overlap NO es desperdicio - es **DATA AUGMENTATION** implícito que mejora la capacidad del modelo de entender lenguaje.

El experimento valida empíricamente que el overlap entre ventanas de contexto no solo aumenta la cantidad de muestras de entrenamiento, sino que fundamentalmente mejora la capacidad del modelo para aprender dependencias lingüísticas complejas al exponer cada token a múltiples contextos diferentes durante el entrenamiento.